# 08b — Kiểm tra ảnh, chia split, khung truy vấn, đặc trưng, data card (ViHeritage-MMKG)

Chạy sau `08a` và sau khi 2 người đã chấm `review/image_check_rater{A,B}.csv` (việc tay 1).

| Bước | Nội dung | Mục kế hoạch |
|---|---|---|
| B1 | Độ chính xác liên kết ảnh – thực thể theo `link_method`, Cohen κ; bỏ ảnh bị chấm sai, bỏ độ sâu 1 nếu < 0,85 | 4.5 |
| B2 | Chia triple 70/15/15 theo quan hệ; cặp đối xứng chung split; thực thể val/test phải có trong train | 4.6 |
| B3 | Chia ảnh theo thực thể; pool val/test | 4.6 |
| B4 | Truy vấn: trực tiếp (tên, bí danh) và gián tiếp 1 bước (từ triple val/test); relevance + danh sách lọc | 3.3, 4.6 |
| B5 | Khung để viết câu truy vấn gián tiếp tự nhiên (việc tay 2) | 3.3 |
| B6 | Đặc trưng CLIP ảnh, văn bản thực thể đa ngôn ngữ | 4.7 |
| B7 | Ghi file (định dạng giống bridge DB15K), `make_splits.py`, `split_manifest.json`, DATA_CARD.md, LICENSES.md | 4.6, 8 |

**Đánh giá truy vấn gián tiếp theo kiểu lọc** (giống đánh giá KGC chuẩn): mỗi triple val/test (x, r, a) cho một truy vấn có
đáp án là ảnh của x. Ảnh của các thực thể khác cũng thỏa (·, r, a) trong toàn KG không tính là sai: chúng bị loại khỏi danh sách
xếp hạng của truy vấn đó (`query_filter.parquet`). Nhờ vậy chỉ số đo đúng việc suy ra triple bị giữ lại, mà không phạt đáp án đúng khác.

**Chạy:** Colab (GPU nhanh hơn ở B6, CPU vẫn được) → Run all, khoảng 5–10 phút.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08b_mmkg_split_features', 1, globals())


## B1. Kiểm tra liên kết ảnh – thực thể (việc tay 1)

Nhãn hợp lệ: `dung`, `sai`, `khong_ro`. Độ chính xác = tỉ lệ `dung` trên các ảnh có nhãn khác `khong_ro`, tính theo người chấm A
(và B nếu có), theo từng `link_method`. Cohen κ tính trên 3 nhãn. Ảnh mà **ít nhất một** người chấm `sai` bị loại khỏi dữ liệu.
Nếu độ chính xác của `P373_d1` (ảnh lấy ở category con) < 0,85 thì bỏ mọi ảnh `P373_d1`.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08b_mmkg_split_features', 3, globals())


## B2. Chia triple (70/15/15, chia tầng theo quan hệ)

Đơn vị chia là một triple, riêng quan hệ đối xứng (`P47` giáp ranh, `NEARBY` ở gần) thì một cặp hai chiều là một đơn vị.
Lặp đến khi ổn định: triple val/test nào có thực thể chưa xuất hiện trong train thì chuyển về train (thực thể không có cạnh train thì
nhúng cấu trúc không học được). Cuối cùng assert: không triple val/test nào (kể cả chiều ngược của quan hệ đối xứng) nằm trong train.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08b_mmkg_split_features', 5, globals())


## B3. Chia ảnh theo thực thể và pool

n ≥ 3 ảnh: 1 test, 1 val, còn lại train; n = 2: 1 train, 1 test; n = 1: ảnh vào test (thực thể không có đặc trưng ảnh lúc huấn luyện).
Ảnh P18 (ảnh đại diện trên Wikidata) ưu tiên ở lại train. Đặc trưng thị giác của thực thể (ở 09) = trung bình CLIP chuẩn hóa L2 của ảnh
train, đúng chính sách `train_image_CLIP_mean_L2` của DB15K. Pool val = mọi ảnh val, pool test = mọi ảnh test.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08b_mmkg_split_features', 7, globals())


## B4. Truy vấn, relevance, danh sách lọc

- **Trực tiếp**: mỗi thực thể có ảnh trong pool của split. Câu = tên tiếng Việt; cách nói thứ hai = bí danh tiếng Việt đầu tiên
  khác tên (nếu có); bản tiếng Anh = nhãn tiếng Anh. Không thêm chữ mẫu nào.
- **Gián tiếp 1 bước**: mỗi triple val/test (h, r, t) với đầu có ảnh trong pool của split. Đáp án là h thì `prediction_direction = head`
  (anchor = t); đáp án là t thì `tail` (anchor = h). Các truy vấn cùng (quan hệ, anchor, chiều) chung một nhóm câu chữ (`text_group`),
  nên người chỉ viết câu một lần cho mỗi nhóm.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08b_mmkg_split_features', 9, globals())


## B5. Khung câu truy vấn gián tiếp (việc tay 2)

`review/queries_to_write.csv`: mỗi dòng là một nhóm (quan hệ, anchor, chiều). Người viết điền `text_vi_1`, `text_vi_2` (hai cách
nói tiếng Việt tự nhiên, khác nhau thật sự) và `text_en`. Quy tắc:
1. Không chứa tên đáp án (cột `answers_example` chỉ để hiểu nghĩa, không chép vào câu).
2. Phải nhắc tới anchor (tên hoặc cách gọi khác) và ý của quan hệ, để câu đúng với mọi đáp án của nhóm.
3. Không dùng một khuôn chung cho mọi nhóm; cột `hint` chỉ gợi ý ý nghĩa.
Một người khác đọc lại và ghi tên vào `reviewed_by`. Notebook 09 đọc file này (đổi tên thành `queries_text.csv` khi xong).

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08b_mmkg_split_features', 11, globals())


## B6. Đặc trưng (đóng băng, trích một lần)

- Ảnh: `clip-ViT-B-32` (CLIP ViT-B/32 gốc), 512 chiều, chuẩn hóa L2. Mọi ảnh (cả val/test) được trích một lần; 09 chỉ lấy trung bình
  ảnh **train** làm đặc trưng thị giác của thực thể.
- Văn bản thực thể: `clip-ViT-B-32-multilingual-v1` (có tiếng Việt, cùng không gian với ảnh CLIP gốc) trên "tên. mô tả. mở bài Wikipedia".
Truy vấn được mã hóa ở 09, sau khi có câu chữ.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08b_mmkg_split_features', 13, globals())


## B7. Ghi file, mã tạo split, data card, giấy phép

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08b_mmkg_split_features', 15, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('08b_mmkg_split_features', 16, globals())
